In [1]:
import gc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils.utils import printmessage
import glob
import tqdm
import gc

In [2]:
plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [4]:
agat_path = "/extdata4/baeklab/Hyeonseo/m6A/anno/agat_merged.gff"
columns = ["chr", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
agat=pd.read_csv(agat_path, sep="\t", header=None, names=columns, skiprows=1)
print(agat)

           chr         source      feature     start       end score strand  \
0         chrX         havana         gene    253743    255091     .      +   
1         chrX         havana   transcript    253743    255091     .      +   
2         chrX         havana         exon    253743    253846     .      +   
3         chrX         havana         exon    254937    255091     .      +   
4         chrX  havana_tagene         gene    267677    276210     .      -   
...        ...            ...          ...       ...       ...   ...    ...   
8201054  chr22         havana  start_codon  50782292  50782294     .      -   
8201055  chr22  havana_tagene         gene  50797082  50798991     .      -   
8201056  chr22  havana_tagene   transcript  50797082  50798991     .      -   
8201057  chr22  havana_tagene         exon  50797082  50797504     .      -   
8201058  chr22  havana_tagene         exon  50798788  50798991     .      -   

        frame                                      

In [5]:
def gff_attribute_to_dict(attribute):
    attribute = attribute
    attribute_dict = {}
    for attr in attribute.split(';'):
        try:
            key, value = attr.split('=')
        except:
            print(attribute)
            print(attr)
            raise ValueError
        key = key.strip()
        attribute_dict[key] = value
    return attribute_dict

In [6]:
agat[["start", "end"]] = agat[["start", "end"]].astype(int)
agat["attribute_dict"] = agat["attribute"].apply(gff_attribute_to_dict)
print(agat)

           chr         source      feature     start       end score strand  \
0         chrX         havana         gene    253743    255091     .      +   
1         chrX         havana   transcript    253743    255091     .      +   
2         chrX         havana         exon    253743    253846     .      +   
3         chrX         havana         exon    254937    255091     .      +   
4         chrX  havana_tagene         gene    267677    276210     .      -   
...        ...            ...          ...       ...       ...   ...    ...   
8201054  chr22         havana  start_codon  50782292  50782294     .      -   
8201055  chr22  havana_tagene         gene  50797082  50798991     .      -   
8201056  chr22  havana_tagene   transcript  50797082  50798991     .      -   
8201057  chr22  havana_tagene         exon  50797082  50797504     .      -   
8201058  chr22  havana_tagene         exon  50798788  50798991     .      -   

        frame                                      

In [7]:
print(agat["feature"].value_counts())
agat["transcript_id"] = agat["attribute_dict"].apply(lambda x: x.get("transcript_id", None))
print(agat["transcript_id"].nunique())

feature
exon               4034985
CDS                2379896
transcript          542608
five_prime_UTR      251894
start_codon         210417
three_prime_utr     204960
stop_codon          204215
five_prime_utr      165446
three_prime_UTR     127541
gene                 78969
Selenocysteine         128
Name: count, dtype: int64
542609


In [8]:
agat["Parent"] = agat["attribute_dict"].apply(lambda x: x.get("Parent", None))
print(agat)

           chr         source      feature     start       end score strand  \
0         chrX         havana         gene    253743    255091     .      +   
1         chrX         havana   transcript    253743    255091     .      +   
2         chrX         havana         exon    253743    253846     .      +   
3         chrX         havana         exon    254937    255091     .      +   
4         chrX  havana_tagene         gene    267677    276210     .      -   
...        ...            ...          ...       ...       ...   ...    ...   
8201054  chr22         havana  start_codon  50782292  50782294     .      -   
8201055  chr22  havana_tagene         gene  50797082  50798991     .      -   
8201056  chr22  havana_tagene   transcript  50797082  50798991     .      -   
8201057  chr22  havana_tagene         exon  50797082  50797504     .      -   
8201058  chr22  havana_tagene         exon  50798788  50798991     .      -   

        frame                                      

In [9]:
def determine_coding(attribute_dict):
    transcript_id = attribute_dict.get("transcript_id", "XXXX")
    if transcript_id.startswith("NM") or transcript_id.startswith("XM"):
        return True
    transcript_biotype = attribute_dict.get("transcript_biotype", "XXXX")
    if transcript_biotype=="protein_coding":
        return True
    merged_gene_id = attribute_dict.get("merged_gene_id", "")
    if "NM_" in merged_gene_id or "XM_" in merged_gene_id:
        return True
    return False


In [10]:
agat_transcript = agat[agat["feature"]=="transcript"].copy()
agat_transcript["coding"] = agat_transcript["attribute_dict"].apply(determine_coding)
print(agat_transcript)
coding_dict = dict(zip(agat_transcript["transcript_id"], agat_transcript["coding"]))

           chr         source     feature     start       end score strand  \
1         chrX         havana  transcript    253743    255091     .      +   
5         chrX  havana_tagene  transcript    267677    276210     .      -   
10        chrX         havana  transcript    276322    303353     .      +   
31        chrX         havana  transcript    276324    291537     .      +   
37        chrX         havana  transcript    276353    291629     .      +   
...        ...            ...         ...       ...       ...   ...    ...   
8201010  chr22         havana  transcript  50771210  50783630     .      -   
8201027  chr22         havana  transcript  50775771  50783600     .      -   
8201044  chr22         havana  transcript  50777660  50783630     .      -   
8201048  chr22         havana  transcript  50782234  50783045     .      -   
8201056  chr22  havana_tagene  transcript  50797082  50798991     .      -   

        frame                                          attribut

In [11]:
agat_gene = agat[agat["feature"]=="gene"].copy()
print(agat_gene)

           chr          source feature     start       end score strand frame  \
0         chrX          havana    gene    253743    255091     .      +     .   
4         chrX   havana_tagene    gene    267677    276210     .      -     .   
9         chrX  ensembl_havana    gene    276322    303356     .      +     .   
293       chrX   havana_tagene    gene    278946    281825     .      -     .   
297       chrX  ensembl_havana    gene    303346    318798     .      -     .   
...        ...             ...     ...       ...       ...   ...    ...   ...   
8199060  chr22          havana    gene  50740593  50743520     .      -     .   
8199064  chr22          havana    gene  50754675  50755434     .      -     .   
8199067  chr22  ensembl_havana    gene  50756831  50801309     .      +     .   
8199617  chr22  ensembl_havana    gene  50767501  50783667     .      -     .   
8201055  chr22   havana_tagene    gene  50797082  50798991     .      -     .   

                           

In [13]:
def get_id(attribute_dict):
    id_main = attribute_dict["gene_id"]
    id_merged = attribute_dict.get("merged_gene_id", None)
    id_all = [id_main]
    if id_merged is not None:
        id_merged = id_merged.split(",")
        id_all+=id_merged
    id_ensembl = [x for x in id_all if x.startswith("ENSG")]
    id_refseq = [x for x in id_all if not x.startswith("ENSG")]
    if len(id_ensembl)==0:
        id_ensembl = None
    else:
        id_ensembl = id_ensembl[0]
    if len(id_refseq)==0:
        id_refseq = None
    else:
        id_refseq = id_refseq[0]
    return id_ensembl, id_refseq

In [14]:
agat_gene[['ensembl_id', 'refseq_id']] = agat_gene["attribute_dict"].apply(lambda x: pd.Series(get_id(x)))
print(agat_gene)

           chr          source feature     start       end score strand frame  \
0         chrX          havana    gene    253743    255091     .      +     .   
4         chrX   havana_tagene    gene    267677    276210     .      -     .   
9         chrX  ensembl_havana    gene    276322    303356     .      +     .   
293       chrX   havana_tagene    gene    278946    281825     .      -     .   
297       chrX  ensembl_havana    gene    303346    318798     .      -     .   
...        ...             ...     ...       ...       ...   ...    ...   ...   
8199060  chr22          havana    gene  50740593  50743520     .      -     .   
8199064  chr22          havana    gene  50754675  50755434     .      -     .   
8199067  chr22  ensembl_havana    gene  50756831  50801309     .      +     .   
8199617  chr22  ensembl_havana    gene  50767501  50783667     .      -     .   
8201055  chr22   havana_tagene    gene  50797082  50798991     .      -     .   

                           

In [16]:
## IF gene_name is NA, use gene of attribute_dict
agat_gene["gene_name"] = agat_gene.apply(lambda x: x["attribute_dict"].get("gene_name",None), axis=1)
agat_gene["gene_name"] = agat_gene.apply(lambda x: x["attribute_dict"].get("gene",None) if x["gene_name"] is None else x["gene_name"], axis=1)
agat_gene["gene_name"] = agat_gene.apply(lambda x: x["refseq_id"] if x["gene_name"] is None else x["gene_name"], axis=1)
agat_gene["gene_name"] = agat_gene.apply(lambda x: x["ensembl_id"] if x["gene_name"] is None else x["gene_name"], axis=1)
print(agat_gene[agat_gene["gene_name"].isna()])

Empty DataFrame
Columns: [chr, source, feature, start, end, score, strand, frame, attribute, attribute_dict, transcript_id, Parent, gene_name, ensembl_id, refseq_id]
Index: []


In [25]:
## Print duplicated ensemble_id
print(agat_gene[agat_gene["ensembl_id"].duplicated() & agat_gene["ensembl_id"].notna()])


Empty DataFrame
Columns: [chr, source, feature, start, end, score, strand, frame, attribute, attribute_dict, transcript_id, Parent, gene_name, ensembl_id, refseq_id]
Index: []


In [26]:
## Print duplicated refseq_id
print(agat_gene[agat_gene["refseq_id"].duplicated() & agat_gene["refseq_id"].notna()])


Empty DataFrame
Columns: [chr, source, feature, start, end, score, strand, frame, attribute, attribute_dict, transcript_id, Parent, gene_name, ensembl_id, refseq_id]
Index: []


In [27]:
agat_gene["gene_id"] = agat_gene["refseq_id"]
agat_gene["gene_id"] = agat_gene["gene_id"].fillna(agat_gene["ensembl_id"])
print(agat_gene[agat_gene["gene_id"].isna()])


Empty DataFrame
Columns: [chr, source, feature, start, end, score, strand, frame, attribute, attribute_dict, transcript_id, Parent, gene_name, ensembl_id, refseq_id, gene_id]
Index: []


In [29]:
agat_gene["ID"] = agat_gene["attribute_dict"].apply(lambda x: x.get("ID", None))

In [30]:
print(agat_gene)

           chr          source feature     start       end score strand frame  \
0         chrX          havana    gene    253743    255091     .      +     .   
4         chrX   havana_tagene    gene    267677    276210     .      -     .   
9         chrX  ensembl_havana    gene    276322    303356     .      +     .   
293       chrX   havana_tagene    gene    278946    281825     .      -     .   
297       chrX  ensembl_havana    gene    303346    318798     .      -     .   
...        ...             ...     ...       ...       ...   ...    ...   ...   
8199060  chr22          havana    gene  50740593  50743520     .      -     .   
8199064  chr22          havana    gene  50754675  50755434     .      -     .   
8199067  chr22  ensembl_havana    gene  50756831  50801309     .      +     .   
8199617  chr22  ensembl_havana    gene  50767501  50783667     .      -     .   
8201055  chr22   havana_tagene    gene  50797082  50798991     .      -     .   

                           

In [31]:
id_to_id_dict = dict(zip(agat_gene["ID"], agat_gene["gene_id"]))

In [32]:
id_to_name_dict = dict(zip(agat_gene["ID"], agat_gene["gene_name"]))

In [33]:
print(agat_transcript["coding"].value_counts())

coding
False    342480
True     200128
Name: count, dtype: int64


In [34]:
def make_refflat(df):
    refflat_dict = {"transcript_id":[], "gene_id":[], "gene_name":[], "chr":[], "strand":[], "txStart":[], "txEnd":[], "cdsStart":[], "cdsEnd":[], "exonCount":[], "exonStarts":[], "exonEnds":[], "coding": []}
    for transcript_id, group in tqdm.tqdm(df, total=df.ngroups):
        refflat_dict["transcript_id"].append(transcript_id)
        id = group[group["feature"]=="transcript"]["Parent"].dropna().unique()
        try: 
            assert len(id)==1
        except:
            print(group)
            raise ValueError
        id = id[0]
        gene_name = id_to_name_dict[id]
        gene_id = id_to_id_dict[id]
        coding = coding_dict[transcript_id]
        
        refflat_dict["gene_id"].append(gene_id)
        refflat_dict["gene_name"].append(gene_name)
        refflat_dict["chr"].append(group["chr"].values[0])
        refflat_dict["strand"].append(group["strand"].values[0])
        start = group["start"].min()
        end = group["end"].max()
        refflat_dict["txStart"].append(start)
        refflat_dict["txEnd"].append(end)
        if group[group["feature"]=="CDS"].shape[0]==0:
            refflat_dict["cdsStart"].append(end)
            refflat_dict["cdsEnd"].append(end)
        else:
            refflat_dict["cdsStart"].append(group[group["feature"]=="CDS"]["start"].min())
            refflat_dict["cdsEnd"].append(group[group["feature"]=="CDS"]["end"].max())
        refflat_dict["exonCount"].append(group[group["feature"]=="exon"].shape[0])
        refflat_dict["exonStarts"].append(",".join(group[group["feature"]=="exon"]["start"].sort_values().astype(str).values)+",")
        refflat_dict["exonEnds"].append(",".join(group[group["feature"]=="exon"]["end"].sort_values().astype(str).values)+",")
        refflat_dict["coding"].append(coding)
    refflat = pd.DataFrame(refflat_dict)
    return refflat



In [35]:
agat_exoncds = agat[agat["feature"].isin(["transcript","exon", "CDS"])]
agat_exoncds = agat_exoncds.groupby("transcript_id")

In [ ]:
agat_refflat = make_refflat(agat_exoncds)

 24%|███████████████████████████████████▉                                                                                                                  | 129804/542608 [05:50<17:39, 389.74it/s]

In [ ]:
agat_refflat.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/agat_refflat.pkl")

In [38]:
print(agat_refflat)

                    transcript_id     gene_id   gene_name    chr strand  \
0                 ENST00000000233  ARF5,FSCN3        ARF5   chr7      +   
1                 ENST00000000442       ESRRA       ESRRA  chr11      +   
2                 ENST00000001008       FKBP4       FKBP4  chr12      +   
3                 ENST00000002125     NDUFAF7     NDUFAF7   chr2      +   
4                 ENST00000002165       FUCA2       FUCA2   chr6      -   
...                           ...         ...         ...    ...    ...   
542603  unassigned_transcript_995  TRH-GTG1-5  TRH-GTG1-5   chr6      +   
542604  unassigned_transcript_996  TRT-AGT6-1  TRT-AGT6-1   chr6      +   
542605  unassigned_transcript_997  TRI-AAT5-2  TRI-AAT5-2   chr6      -   
542606  unassigned_transcript_998  TRV-CAC6-1  TRV-CAC6-1   chr6      -   
542607  unassigned_transcript_999  TRS-CGA2-1  TRS-CGA2-1   chr6      +   

          txStart      txEnd   cdsStart     cdsEnd  exonCount  \
0       127588411  127591700  1275

In [45]:
print(agat_refflat[agat_refflat["transcript_id"].str.startswith("unassigned_transcript")])

                     transcript_id       gene_id               gene_name  \
538718     unassigned_transcript_1        WASH7P        WASH7P,MIR6859-1   
538719    unassigned_transcript_10  LOC124903818  MIR200B,MIR200A,MIR429   
538720   unassigned_transcript_100     MIR3116-1               MIR3116-1   
538721  unassigned_transcript_1000    TRR-ACG2-2              TRR-ACG2-2   
538722  unassigned_transcript_1001    TRR-ACG2-3              TRR-ACG2-3   
...                            ...           ...                     ...   
542603   unassigned_transcript_995    TRH-GTG1-5              TRH-GTG1-5   
542604   unassigned_transcript_996    TRT-AGT6-1              TRT-AGT6-1   
542605   unassigned_transcript_997    TRI-AAT5-2              TRI-AAT5-2   
542606   unassigned_transcript_998    TRV-CAC6-1              TRV-CAC6-1   
542607   unassigned_transcript_999    TRS-CGA2-1              TRS-CGA2-1   

         chr strand   txStart     txEnd  cdsStart    cdsEnd  exonCount  \
538718  chr1 

In [43]:
print(agat_gene[agat_gene["gene_id"].str.startswith("ARF5")]["attribute_dict"].values[0])

{'ID': 'ENSG00000004059', 'db_xref': 'GeneID:381,HGNC:HGNC:658,MIM:103188', 'description': 'ADP ribosylation factor 5', 'gbkey': 'Gene', 'gene': 'ARF5', 'gene_biotype': 'protein_coding', 'gene_id': 'ENSG00000004059', 'gene_name': 'ARF5,FSCN3', 'gene_source': 'ensembl_havana', 'gene_version': '11,10', 'merged_ID': 'ARF5,ENSG00000106328', 'merged_gene_id': 'ARF5,ENSG00000106328', 'transcript_id': '""'}


In [47]:
refflat_columns = ["gene_id", "transcript_id", "chr", "strand", "txStart", "txEnd", "cdsStart", "cdsEnd", "exonCount", "exonStarts", "exonEnds"]

In [48]:
agat_refflat[refflat_columns].to_csv("/extdata4/baeklab/Hyeonseo/m6A/anno/agat_refflat.txt", sep="\t", index=False, header=False)

In [49]:
agat_refflat.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/agat_refflat.pkl")